In [1]:
# raw"..." = pega o caminho exatamente como está, sem interpretar as barras \
pasta = raw"C:\Users\Saidk\OneDrive\Desktop\Mestrado\optimization2026\materiais\AULA06\jsplib_subset\instances"

# joinpath junta a pasta + o nome do arquivo com a barra certa do sistema
caminho = joinpath(pasta, "ft06")

linhas = readlines(caminho)
for (i, linha) in enumerate(linhas)
    println(i, ": ", linha)
end

1: #+++++++++++++++++++++++++++++
2: # instance ft06
3: #+++++++++++++++++++++++++++++
4: # Fisher and Thompson 6x6 instance, alternate name (mt06)
5: 6 6
6: 2  1  0  3  1  6  3  7  5  3  4  6
7: 1  8  2  5  4 10  5 10  0 10  3  4
8: 2  5  3  4  5  8  0  9  1  1  4  7
9: 1  5  0  5  2  5  3  3  4  8  5  9
10: 2  9  1  3  4  5  5  4  0  3  3  1
11: 1  3  3  3  5  9  0 10  4  4  2  1


In [2]:
# 1) limpa as pontas de cada linha (tira espaços e aquele \r invisível)
limpas = strip.(linhas)

# 2) fica só com as linhas úteis: fora as de comentário (#) e as vazias
uteis = filter(l -> !startswith(l, "#") && !isempty(l), limpas)

# 3) a primeira linha útil traz as dimensões: "6 6"
dimensoes = parse.(Int, split(uteis[1]))
n_tarefas  = dimensoes[1]
n_maquinas = dimensoes[2]

println("tarefas:  ", n_tarefas)
println("máquinas: ", n_maquinas)
println("linhas de tarefa pra processar: ", length(uteis) - 1)

tarefas:  6
máquinas: 6
linhas de tarefa pra processar: 6


In [3]:
tarefas_linhas = uteis[2:end]   # as 6 linhas de dados (pula a linha "6 6" das dimensões)

# receitas: uma lista onde cada item é a receita de uma tarefa;
# cada receita é uma lista de pares (máquina, duração)
receitas = Vector{Vector{Tuple{Int,Int}}}()

for linha in tarefas_linhas
    numeros = parse.(Int, split(linha))   # todos os números da linha, já como inteiros
    operacoes = Tuple{Int,Int}[]           # a receita desta tarefa (começa vazia)
    for k in 1:2:length(numeros)           # conta pulando de 2 em 2: 1, 3, 5, ...
        maquina = numeros[k]
        duracao = numeros[k+1]
        push!(operacoes, (maquina, duracao))
    end
    push!(receitas, operacoes)
end

# imprime pra conferir com os próprios olhos
for (j, operacoes) in enumerate(receitas)
    print("tarefa ", j, ": ")
    for (maquina, duracao) in operacoes
        print("M", maquina, "(", duracao, ")  ")
    end
    println()
end

tarefa 1: M2(1)  M0(3)  M1(6)  M3(7)  M5(3)  M4(6)  
tarefa 2: M1(8)  M2(5)  M4(10)  M5(10)  M0(10)  M3(4)  
tarefa 3: M2(5)  M3(4)  M5(8)  M0(9)  M1(1)  M4(7)  
tarefa 4: M1(5)  M0(5)  M2(5)  M3(3)  M4(8)  M5(9)  
tarefa 5: M2(9)  M1(3)  M4(5)  M5(4)  M0(3)  M3(1)  
tarefa 6: M1(3)  M3(3)  M5(9)  M0(10)  M4(4)  M2(1)  


In [4]:
function ler_instancia(caminho)
    linhas = readlines(caminho)
    limpas = strip.(linhas)
    uteis  = filter(l -> !startswith(l, "#") && !isempty(l), limpas)

    dimensoes  = parse.(Int, split(uteis[1]))
    n_tarefas  = dimensoes[1]
    n_maquinas = dimensoes[2]

    receitas = Vector{Vector{Tuple{Int,Int}}}()
    for linha in uteis[2:end]
        numeros   = parse.(Int, split(linha))
        operacoes = Tuple{Int,Int}[]
        for k in 1:2:length(numeros)
            push!(operacoes, (numeros[k], numeros[k+1]))
        end
        push!(receitas, operacoes)
    end

    return (n_tarefas = n_tarefas, n_maquinas = n_maquinas, receitas = receitas)
end

ler_instancia (generic function with 1 method)

In [5]:
inst = ler_instancia(joinpath(pasta, "ft06"))
println("ft06  -> ", inst.n_tarefas, " tarefas, ", inst.n_maquinas, " máquinas")
println("receita da tarefa 1: ", inst.receitas[1])

inst_la01 = ler_instancia(joinpath(pasta, "la01"))
println("la01  -> ", inst_la01.n_tarefas, " tarefas, ", inst_la01.n_maquinas, " máquinas")
println("receita da tarefa 1: ", inst_la01.receitas[1])

ft06  -> 6 tarefas, 6 máquinas
receita da tarefa 1: [(2, 1), (0, 3), (1, 6), (3, 7), (5, 3), (4, 6)]
la01  -> 10 tarefas, 5 máquinas
receita da tarefa 1: [(1, 21), (0, 53), (4, 95), (3, 55), (2, 34)]


In [6]:
using JuMP
using HiGHS

inst = ler_instancia(joinpath(pasta, "ft06"))
n = inst.n_tarefas    # nº de tarefas (6)
m = inst.n_maquinas   # nº de máquinas = nº de operações por tarefa (6)

modelo = Model(HiGHS.Optimizer)

# s[j,k] = instante em que COMEÇA a k-ésima operação da tarefa j
@variable(modelo, s[1:n, 1:m] >= 0)

# Cmax = o makespan (a borda direita do Gantt)
@variable(modelo, Cmax >= 0)

println("modelo criado com ", num_variables(modelo), " variáveis")

modelo criado com 37 variáveis


In [7]:
# precedência: dentro de cada tarefa, a próxima operação só começa
# depois que a anterior termina (começo + duração)
n_prec = 0
for j in 1:n
    for k in 1:(m-1)
        dur_k = inst.receitas[j][k][2]   # duração da k-ésima operação da tarefa j
        @constraint(modelo, s[j, k+1] >= s[j, k] + dur_k)
        n_prec += 1
    end
end
println("restrições de precedência criadas: ", n_prec)

# só pra você VER a regra da tarefa 1 em palavras:
println("\nprecedência da tarefa 1:")
for k in 1:(m-1)
    dur_k = inst.receitas[1][k][2]
    println("  op", k+1, " começa só depois de op", k, " terminar (op", k, " dura ", dur_k, ")")
end

restrições de precedência criadas: 30

precedência da tarefa 1:
  op2 começa só depois de op1 terminar (op1 dura 1)
  op3 começa só depois de op2 terminar (op2 dura 3)
  op4 começa só depois de op3 terminar (op3 dura 6)
  op5 começa só depois de op4 terminar (op4 dura 7)
  op6 começa só depois de op5 terminar (op5 dura 3)


In [8]:
# Cmax >= fim da última operação de cada tarefa
n_mk = 0
for j in 1:n
    dur_ultima = inst.receitas[j][m][2]   # duração da última (m-ésima) operação da tarefa j
    @constraint(modelo, Cmax >= s[j, m] + dur_ultima)
    n_mk += 1
end
println("restrições de makespan criadas: ", n_mk)

restrições de makespan criadas: 6


In [9]:
# (d1) agrupar as operações por máquina
# cada operação é identificada por (tarefa j, posição k na receita)
ops_por_maquina = Dict{Int, Vector{Tuple{Int,Int}}}()

for j in 1:n
    for k in 1:m
        maq = inst.receitas[j][k][1]        # o NOME da máquina desta operação (0..5)
        if !haskey(ops_por_maquina, maq)
            ops_por_maquina[maq] = Tuple{Int,Int}[]   # 1ª vez que vejo essa máquina: cria lista vazia
        end
        push!(ops_por_maquina[maq], (j, k))           # anexa a operação (tarefa j, posição k)
    end
end

# mostrar quantas operações caem em cada máquina
for maq in sort(collect(keys(ops_por_maquina)))
    ops = ops_por_maquina[maq]
    println("máquina ", maq, ": ", length(ops), " operações  ->  ", ops)
end

máquina 0: 6 operações  ->  [(1, 2), (2, 5), (3, 4), (4, 2), (5, 5), (6, 4)]
máquina 1: 6 operações  ->  [(1, 3), (2, 1), (3, 5), (4, 1), (5, 2), (6, 1)]
máquina 2: 6 operações  ->  [(1, 1), (2, 2), (3, 1), (4, 3), (5, 1), (6, 6)]
máquina 3: 6 operações  ->  [(1, 4), (2, 6), (3, 2), (4, 4), (5, 6), (6, 2)]
máquina 4: 6 operações  ->  [(1, 6), (2, 3), (3, 6), (4, 5), (5, 3), (6, 5)]
máquina 5: 6 operações  ->  [(1, 5), (2, 4), (3, 3), (4, 6), (5, 4), (6, 3)]


In [10]:
# big-M: um número grande o bastante pra "desligar" uma regra.
# A soma de TODAS as durações serve: nenhuma operação pode começar depois disso.
M = sum(inst.receitas[j][k][2] for j in 1:n for k in 1:m)
println("valor de M (soma de todas as durações): ", M)

n_bin = 0
for (maq, ops) in ops_por_maquina          # para cada máquina e sua lista de operações
    for a in 1:length(ops)
        for b in (a+1):length(ops)         # cada PAR de operações da máquina (sem repetir)
            (j1, k1) = ops[a]
            (j2, k2) = ops[b]
            dur1 = inst.receitas[j1][k1][2]
            dur2 = inst.receitas[j2][k2][2]

            # a chavinha deste par: x = 1 -> (j1,k1) vem antes; x = 0 -> a outra vem antes
            x = @variable(modelo, binary = true)

            # regra "(j1,k1) antes": ativa quando x = 1; desligada (M) quando x = 0
            @constraint(modelo, s[j1,k1] + dur1 <= s[j2,k2] + M*(1 - x))
            # regra "(j2,k2) antes": ativa quando x = 0; desligada (M) quando x = 1
            @constraint(modelo, s[j2,k2] + dur2 <= s[j1,k1] + M*x)

            n_bin += 1
        end
    end
end

println("variáveis binárias criadas (pares na mesma máquina): ", n_bin)
println("total de variáveis no modelo agora: ", num_variables(modelo))

valor de M (soma de todas as durações): 197
variáveis binárias criadas (pares na mesma máquina): 90
total de variáveis no modelo agora: 127


In [11]:
@objective(modelo, Min, Cmax)   # minimizar o makespan

optimize!(modelo)               # manda o HiGHS resolver

println("status: ", termination_status(modelo))
println("makespan encontrado: ", objective_value(modelo))

Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline-5 
MIP has 216 rows; 127 cols; 612 nonzeros; 90 integer variables (90 binary)
Coefficient ranges:
  Matrix  [1e+00, 2e+02]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 2e+02]
Presolving model
216 rows, 127 cols, 612 nonzeros 0s
126 rows, 127 cols, 342 nonzeros 0s
126 rows, 127 cols, 342 nonzeros 0s
Presolve reductions: rows 126(-90); columns 127(-0); nonzeros 342(-270) 

Solving MIP model with:
   126 rows
   127 cols (90 binary, 0 integer, 0 implied int., 37 continuous, 0 domain fixed)
   342 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Tr

In [12]:
# ótimos conhecidos (confira depois com o instances.json do professor)
otimos = Dict("ft06"=>55, "ft10"=>930, "ft20"=>1165,
              "la01"=>666, "la02"=>655, "la03"=>597, "la04"=>590,
              "abz5"=>1234, "abz6"=>943, "orb01"=>1059)

function resolver(nome; tempo_limite = 60)
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)   # não imprime o log gigante

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)

    # precedência
    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    # makespan
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end
    # disjunção big-M
    M = sum(inst.receitas[j][k][2] for j in 1:n for k in 1:m)
    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        maq = inst.receitas[j][k][1]
        push!(get!(ops, maq, Tuple{Int,Int}[]), (j,k))
    end
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + M*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + M*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    ok  = has_values(modelo)
    mk  = ok ? round(Int, objective_value(modelo)) : missing
    opt = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing   # % acima do ótimo
    return (nome=nome, n=n, m=m, makespan=mk, otimo=opt,
            dist=dist, tempo=round(solve_time(modelo), digits=2),
            status=termination_status(modelo))
end

# teste na ft06:
resultado = resolver("ft06")
println(resultado)

(nome = "ft06", n = 6, m = 6, makespan = 55, otimo = 55, dist = 0.0, tempo = 0.6, status = OPTIMAL)


In [ ]:
nomes = ["ft06","la01","la02","la03","la04","ft20","ft10","abz5","abz6","orb01"]

resultados = []
for nome in nomes
    r = resolver(nome; tempo_limite = 60)
    push!(resultados, r)
    println(rpad(r.nome,6), " | ", r.n, "x", r.m,
            " | mk=", r.makespan, " | ótimo=", r.otimo,
            " | +", r.dist, "% | ", r.tempo, "s | ", r.status)
end

ft06  

In [ ]:
nomes = ["ft06","la01","la02","la03","la04","ft20","ft10","abz5","abz6","orb01"]

resultados = []
for (i, nome) in enumerate(nomes)
    println("(", i, "/", length(nomes), ") resolvendo ", nome, "...")
    flush(stdout)                       # força aparecer na hora
    r = resolver(nome; tempo_limite = 120)
    push!(resultados, r)
    println("   -> mk=", r.makespan, " | ótimo=", r.otimo,
            " | +", r.dist, "% | ", r.tempo, "s | ", r.status)
    flush(stdout)
end

In [12]:
function resolver(nome; tempo_limite = 60)
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)

    # precedência
    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    # makespan
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end
    # disjunção big-M
    M = sum(inst.receitas[j][k][2] for j in 1:n for k in 1:m)
    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        maq = inst.receitas[j][k][1]
        push!(get!(ops, maq, Tuple{Int,Int}[]), (j,k))
    end
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + M*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + M*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    # --- resultados (com as 2 colunas novas: gap e binárias) ---
    ok   = has_values(modelo)
    mk   = ok ? round(Int, objective_value(modelo)) : missing
    opt  = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing
    g    = relative_gap(modelo)
    gap  = (ok && isfinite(g)) ? round(100*g, digits=1) : missing
    bins = count(is_binary, all_variables(modelo))
    return (nome=nome, n=n, m=m, makespan=mk, otimo=opt, dist=dist,
            gap=gap, binarias=bins,
            tempo=round(solve_time(modelo), digits=2),
            status=termination_status(modelo))
end

resolver (generic function with 1 method)

In [12]:
otimos = Dict("ft06"=>55, "ft10"=>930, "ft20"=>1165,
              "la01"=>666, "la02"=>655, "la03"=>597, "la04"=>590,
              "abz5"=>1234, "abz6"=>943, "orb01"=>1059)

Dict{String, Int64} with 10 entries:
  "ft20"  => 1165
  "abz6"  => 943
  "la03"  => 597
  "ft06"  => 55
  "ft10"  => 930
  "abz5"  => 1234
  "orb01" => 1059
  "la02"  => 655
  "la01"  => 666
  "la04"  => 590

In [ ]:
restantes = ["la01","la02","la03","ft20","ft10","abz5","orb01"]
res240 = []
for (i,nome) in enumerate(restantes)
    println("(",i,"/",length(restantes),") ",nome,"..."); flush(stdout)
    r = resolver(nome; tempo_limite = 240)
    push!(res240, r)
    println("   -> mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)
    flush(stdout)
end

(1/7) la01...
   -> mk=666 | +0.0% | gap=0.0% | 159.77s | OPTIMAL
(2/7) la02...
   -> mk=655 | +0.0% | gap=0.0% | 155.84s | OPTIMAL
(3/7) la03...
   -> mk=597 | +0.0% | gap=16.2% | 240.01s | TIME_LIMIT
(4/7) ft20...
   -> mk=980 | +5.4% | gap=24.3% | 240.01s | TIME_LIMIT
(6/7) abz5...
   -> mk=1239 | +0.4% | gap=16.6% | 240.01s | TIME_LIMIT
(7/7) orb01...
   -> mk=1106 | +4.4% | gap=29.5% | 240.01s | TIME_LIMIT


In [14]:
r = resolver("ft10"; tempo_limite = 240)
println("ft10 -> mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)

ft10 -> mk=980 | +5.4% | gap=24.3% | 240.01s | TIME_LIMIT


In [15]:
function resolver(nome; tempo_limite = 60)
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)

    # precedência
    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    # makespan
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end

    # disjunção big-M APERTADO (um M por par, não um M global)
    total_tarefa = [sum(inst.receitas[j][k][2] for k in 1:m) for j in 1:n]

    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        maq = inst.receitas[j][k][1]
        push!(get!(ops, maq, Tuple{Int,Int}[]), (j,k))
    end
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        Mpar = total_tarefa[j1] + total_tarefa[j2]   # <- M do par (bem menor que o global)
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + Mpar*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + Mpar*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    ok   = has_values(modelo)
    mk   = ok ? round(Int, objective_value(modelo)) : missing
    opt  = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing
    g    = relative_gap(modelo)
    gap  = (ok && isfinite(g)) ? round(100*g, digits=1) : missing
    bins = count(is_binary, all_variables(modelo))
    return (nome=nome, n=n, m=m, makespan=mk, otimo=opt, dist=dist,
            gap=gap, binarias=bins,
            tempo=round(solve_time(modelo), digits=2),
            status=termination_status(modelo))
end

resolver (generic function with 1 method)

In [ ]:
r = resolver("la03"; tempo_limite = 240)
println("la03 (M apertado) -> mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)

In [13]:
function resolver(nome; tempo_limite = 60)
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)

    # precedência
    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    # makespan
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end

    # disjunção big-M APERTADO (um M por par)
    total_tarefa = [sum(inst.receitas[j][k][2] for k in 1:m) for j in 1:n]

    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        maq = inst.receitas[j][k][1]
        push!(get!(ops, maq, Tuple{Int,Int}[]), (j,k))
    end

    Ms_do_par = Int[]                          # coleta os M de cada par
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        Mpar = total_tarefa[j1] + total_tarefa[j2]
        push!(Ms_do_par, Mpar)
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + Mpar*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + Mpar*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    ok   = has_values(modelo)
    mk   = ok ? round(Int, objective_value(modelo)) : missing
    opt  = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing
    g    = relative_gap(modelo)
    gap  = (ok && isfinite(g)) ? round(100*g, digits=1) : missing
    bins = count(is_binary, all_variables(modelo))

    return (nome=nome, n=n, m=m, makespan=mk, otimo=opt, dist=dist,
            gap=gap, binarias=bins,
            M_frouxo = sum(total_tarefa),          # o M global antigo (soma de tudo)
            M_apertado_max = maximum(Ms_do_par),   # maior M por par
            M_apertado_min = minimum(Ms_do_par),   # menor M por par
            tempo = round(solve_time(modelo), digits=2),
            status = termination_status(modelo))
end

resolver (generic function with 1 method)

In [14]:
r = resolver("la03"; tempo_limite = 240)
println("la03 | mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"%",
        " | M frouxo=",r.M_frouxo,
        " | M apertado: ",r.M_apertado_min," a ",r.M_apertado_max,
        " | ",r.tempo,"s | ",r.status)

la03 | mk=missing | +missing% | gap=missing% | M frouxo=2383 | M apertado: 316 a 646 | 240.04s | TIME_LIMIT


In [15]:
function resolver(nome; tempo_limite = 60)
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)

    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end

    # --- M SEGURO por par ---
    # teto global da instância = soma de TODAS as durações (nenhum cronograma passa disso)
    horizonte = sum(inst.receitas[j][k][2] for j in 1:n for k in 1:m)

    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        maq = inst.receitas[j][k][1]
        push!(get!(ops, maq, Tuple{Int,Int}[]), (j,k))
    end

    Ms_do_par = Int[]
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        # M do par = horizonte menos o trabalho que NÃO pertence a este par.
        # é sempre >= atraso possível entre as duas ops (nunca corta solução),
        # mas < horizonte cheio (aperta a prova).
        Mpar = horizonte - (sum(inst.receitas[j1][kk][2] for kk in 1:m) - d1) -
                           (sum(inst.receitas[j2][kk][2] for kk in 1:m) - d2)
        Mpar = max(Mpar, d1 + d2)      # trava de segurança: nunca menor que as duas ops
        push!(Ms_do_par, Mpar)
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + Mpar*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + Mpar*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    ok   = has_values(modelo)
    mk   = ok ? round(Int, objective_value(modelo)) : missing
    opt  = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing
    g    = relative_gap(modelo)
    gap  = (ok && isfinite(g)) ? round(100*g, digits=1) : missing
    bins = count(is_binary, all_variables(modelo))

    return (nome=nome, n=n, m=m, makespan=mk, otimo=opt, dist=dist,
            gap=gap, binarias=bins,
            M_frouxo = horizonte,
            M_apertado_max = maximum(Ms_do_par),
            M_apertado_min = minimum(Ms_do_par),
            tempo = round(solve_time(modelo), digits=2),
            status = termination_status(modelo))
end

resolver (generic function with 1 method)

In [16]:
r = resolver("la03"; tempo_limite = 240)
println("la03 | mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"%",
        " | M frouxo=",r.M_frouxo," | M apertado: ",r.M_apertado_min," a ",r.M_apertado_max,
        " | ",r.tempo,"s | ",r.status)

la03 | mk=597 | +0.0% | gap=0.0% | M frouxo=2383 | M apertado: 1811 a 2190 | 197.23s | OPTIMAL


In [17]:
teimosas = ["la03","ft20","ft10","abz5","orb01"]
res_apertado = []
for (i,nome) in enumerate(teimosas)
    println("(",i,"/",length(teimosas),") ",nome,"..."); flush(stdout)
    r = resolver(nome; tempo_limite = 240)
    push!(res_apertado, r)
    println("   -> mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,
            "% | Mapert: ",r.M_apertado_min,"-",r.M_apertado_max," | ",r.tempo,"s | ",r.status)
    flush(stdout)
end

(1/5) la03...
   -> mk=597 | +0.0% | gap=0.0% | Mapert: 1811-2190 | 198.53s | OPTIMAL
(2/5) ft20...
   -> mk=1315 | +12.9% | gap=63.0% | Mapert: 4473-4898 | 240.01s | TIME_LIMIT
(3/5) ft10...
   -> mk=981 | +5.5% | gap=23.7% | Mapert: 3890-4439 | 240.01s | TIME_LIMIT
(4/5) abz5...
   -> mk=1238 | +0.3% | gap=16.9% | Mapert: 6236-6526 | 240.01s | TIME_LIMIT
(5/5) orb01...
   -> mk=1156 | +9.2% | gap=32.6% | Mapert: 4127-4678 | 240.01s | TIME_LIMIT


In [20]:
function resolver(nome; tempo_limite = 60)
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)
    set_optimizer_attribute(modelo, "threads", 0)                  # 0 = usa todos os núcleos do PC
    set_optimizer_attribute(modelo, "mip_heuristic_effort", 0.2)   # + esforço achando boas soluções
    set_optimizer_attribute(modelo, "mip_detect_symmetry", true)   # aproveita simetrias do modelo

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)

    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end

    horizonte = sum(inst.receitas[j][k][2] for j in 1:n for k in 1:m)

    # --- M FINO: por operação, o "mais cedo que começa" e o "mais tarde que termina" ---
    cedo  = zeros(Int, n, m)   # release: trabalho ANTES da operação, na mesma tarefa
    tarde = zeros(Int, n, m)   # deadline: horizonte - trabalho DEPOIS - a própria
    for j in 1:n
        antes = 0
        for k in 1:m
            cedo[j,k] = antes
            antes += inst.receitas[j][k][2]
        end
        depois = 0
        for k in m:-1:1
            dk = inst.receitas[j][k][2]
            tarde[j,k] = horizonte - depois          # mais tarde que ESTA op pode terminar
            depois += dk
        end
    end

    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        maq = inst.receitas[j][k][1]
        push!(get!(ops, maq, Tuple{Int,Int}[]), (j,k))
    end

    Ms_do_par = Int[]
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        # M do par = maior atraso possível entre as duas, nos dois sentidos
        M12 = tarde[j1,k1] - cedo[j2,k2]     # A pode empurrar B
        M21 = tarde[j2,k2] - cedo[j1,k1]     # B pode empurrar A
        Mpar = max(M12, M21, d1 + d2)        # trava de segurança
        push!(Ms_do_par, Mpar)
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + Mpar*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + Mpar*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    ok   = has_values(modelo)
    mk   = ok ? round(Int, objective_value(modelo)) : missing
    opt  = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing
    g    = relative_gap(modelo)
    gap  = (ok && isfinite(g)) ? round(100*g, digits=1) : missing
    bins = count(is_binary, all_variables(modelo))

    return (nome=nome, n=n, m=m, makespan=mk, otimo=opt, dist=dist,
            gap=gap, binarias=bins,
            M_frouxo = horizonte,
            M_apertado_max = maximum(Ms_do_par),
            M_apertado_min = minimum(Ms_do_par),
            tempo = round(solve_time(modelo), digits=2),
            status = termination_status(modelo))
end

resolver (generic function with 1 method)

In [19]:
r = resolver("la03"; tempo_limite = 240)
println("la03 | mk=",r.makespan," | gap=",r.gap,"%",
        " | M apertado: ",r.M_apertado_min," a ",r.M_apertado_max," | ",r.tempo,"s | ",r.status)

la03 | mk=597 | gap=0.0% | M apertado: 2134 a 2383 | 183.25s | OPTIMAL


In [21]:
println("núcleos disponíveis: ", Sys.CPU_THREADS)

núcleos disponíveis: 8


In [22]:
r = resolver("la03"; tempo_limite = 240)
println("la03 | mk=",r.makespan," | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)

la03 | mk=597 | gap=0.0% | 217.25s | OPTIMAL


In [23]:
r = resolver("ft10"; tempo_limite = 240)
println("ft10 (com botões) | mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)

ft10 (com botões) | mk=959 | +3.1% | gap=22.0% | 240.01s | TIME_LIMIT


In [24]:
grandes = ["ft20","ft10","abz5","orb01"]
res_final = []
for (i,nome) in enumerate(grandes)
    println("(",i,"/",length(grandes),") ",nome,"..."); flush(stdout)
    r = resolver(nome; tempo_limite = 240)
    push!(res_final, r)
    println("   -> mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)
    flush(stdout)
end

(1/4) ft20...
   -> mk=1287 | +10.5% | gap=61.3% | 240.01s | TIME_LIMIT
(2/4) ft10...
   -> mk=959 | +3.1% | gap=22.0% | 240.02s | TIME_LIMIT
(3/4) abz5...
   -> mk=1239 | +0.4% | gap=18.4% | 240.01s | TIME_LIMIT
(4/4) orb01...
   -> mk=1109 | +4.7% | gap=29.1% | 240.01s | TIME_LIMIT


In [25]:
# --- resultados M FROUXO (método base), das rodadas de 240s ---
# (nome, n, m, makespan, ótimo, dist%, gap%, tempo, status)
base = [
    ("ft06",  6, 6,   55,   55,  0.0,  0.0,   0.65, "OPTIMAL"),
    ("la01", 10, 5,  666,  666,  0.0,  0.0, 159.77, "OPTIMAL"),
    ("la02", 10, 5,  655,  655,  0.0,  0.0, 155.84, "OPTIMAL"),
    ("la03", 10, 5,  597,  597,  0.0, 16.2, 240.01, "TIME_LIMIT"),
    ("la04", 10, 5,  590,  590,  0.0,  0.0,  88.65, "OPTIMAL"),
    ("abz6", 10,10,  943,  943,  0.0,  0.0,  88.28, "OPTIMAL"),
    ("ft20", 20, 5,  980, 1165,  5.4, 24.3, 240.01, "TIME_LIMIT"),
    ("ft10", 10,10,  980,  930,  5.4, 24.3, 240.01, "TIME_LIMIT"),
    ("abz5", 10,10, 1239, 1234,  0.4, 16.6, 240.01, "TIME_LIMIT"),
    ("orb01",10,10, 1106, 1059,  4.4, 29.5, 240.01, "TIME_LIMIT"),
]

# --- resultados FINAIS (M fino + botões do solver) ---
final = [
    ("ft06",  6, 6,   55,   55,  0.0,  0.0,   0.6, "OPTIMAL"),
    ("la01", 10, 5,  666,  666,  0.0,  0.0,   0.0, "OPTIMAL"),   # rerrode se quiser o tempo exato
    ("la02", 10, 5,  655,  655,  0.0,  0.0,   0.0, "OPTIMAL"),
    ("la03", 10, 5,  597,  597,  0.0,  0.0, 217.25, "OPTIMAL"),
    ("la04", 10, 5,  590,  590,  0.0,  0.0,   0.0, "OPTIMAL"),
    ("abz6", 10,10,  943,  943,  0.0,  0.0,   0.0, "OPTIMAL"),
    ("ft20", 20, 5, 1287, 1165, 10.5, 61.3, 240.01, "TIME_LIMIT"),
    ("ft10", 10,10,  959,  930,  3.1, 22.0, 240.02, "TIME_LIMIT"),
    ("abz5", 10,10, 1239, 1234,  0.4, 18.4, 240.01, "TIME_LIMIT"),
    ("orb01",10,10, 1109, 1059,  4.7, 29.1, 240.01, "TIME_LIMIT"),
]

println("base:  ", length(base), " instâncias")
println("final: ", length(final), " instâncias")

base:  10 instâncias
final: 10 instâncias


In [26]:
using DataFrames

# junta base + final numa tabela comparativa
cols = [:nome, :n, :m, :makespan, :otimo, :dist, :gap, :tempo, :status]
df_base  = DataFrame(NamedTuple{Tuple(cols)}.(base))
df_final = DataFrame(NamedTuple{Tuple(cols)}.(final))

# tabela comparativa: só as colunas que interessam pra ver o efeito
tabela = DataFrame(
    instancia = df_base.nome,
    tam       = string.(df_base.n, "x", df_base.m),
    otimo     = df_base.otimo,
    mk_frouxo   = df_base.makespan,
    gap_frouxo  = df_base.gap,
    mk_final    = df_final.makespan,
    gap_final   = df_final.gap,
    status_final = df_final.status,
)

show(tabela, allrows=true, allcols=true)

10×8 DataFrame
 Row │ instancia  tam     otimo  mk_frouxo  gap_frouxo  mk_final  gap_final  status_final 
     │ String     String  Int64  Int64      Float64     Int64     Float64    String       
─────┼────────────────────────────────────────────────────────────────────────────────────
   1 │ ft06       6x6        55         55         0.0        55        0.0  OPTIMAL
   2 │ la01       10x5      666        666         0.0       666        0.0  OPTIMAL
   3 │ la02       10x5      655        655         0.0       655        0.0  OPTIMAL
   4 │ la03       10x5      597        597        16.2       597        0.0  OPTIMAL
   5 │ la04       10x5      590        590         0.0       590        0.0  OPTIMAL
   6 │ abz6       10x10     943        943         0.0       943        0.0  OPTIMAL
   7 │ ft20       20x5     1165        980        24.3      1287       61.3  TIME_LIMIT
   8 │ ft10       10x10     930        980        24.3       959       22.0  TIME_LIMIT
   9 │ abz5       10x10   

In [28]:
function resolver(nome; tempo_limite = 60, modo = "fino")
    inst = ler_instancia(joinpath(pasta, nome))
    n, m = inst.n_tarefas, inst.n_maquinas

    modelo = Model(HiGHS.Optimizer)
    set_optimizer_attribute(modelo, "time_limit", Float64(tempo_limite))
    set_silent(modelo)
    if modo == "fino"                                   # botões só no modo fino
        set_optimizer_attribute(modelo, "threads", 0)
        set_optimizer_attribute(modelo, "mip_heuristic_effort", 0.2)
        set_optimizer_attribute(modelo, "mip_detect_symmetry", true)
    end

    @variable(modelo, s[1:n, 1:m] >= 0)
    @variable(modelo, Cmax >= 0)
    for j in 1:n, k in 1:(m-1)
        @constraint(modelo, s[j,k+1] >= s[j,k] + inst.receitas[j][k][2])
    end
    for j in 1:n
        @constraint(modelo, Cmax >= s[j,m] + inst.receitas[j][m][2])
    end

    horizonte = sum(inst.receitas[j][k][2] for j in 1:n for k in 1:m)
    cedo  = zeros(Int, n, m); tarde = zeros(Int, n, m)
    for j in 1:n
        antes = 0
        for k in 1:m; cedo[j,k] = antes; antes += inst.receitas[j][k][2]; end
        depois = 0
        for k in m:-1:1; tarde[j,k] = horizonte - depois; depois += inst.receitas[j][k][2]; end
    end

    ops = Dict{Int, Vector{Tuple{Int,Int}}}()
    for j in 1:n, k in 1:m
        push!(get!(ops, inst.receitas[j][k][1], Tuple{Int,Int}[]), (j,k))
    end

    Ms = Int[]
    for (maq, lista) in ops, a in 1:length(lista), b in (a+1):length(lista)
        (j1,k1), (j2,k2) = lista[a], lista[b]
        d1, d2 = inst.receitas[j1][k1][2], inst.receitas[j2][k2][2]
        Mpar = modo == "frouxo" ? horizonte :
               max(tarde[j1,k1]-cedo[j2,k2], tarde[j2,k2]-cedo[j1,k1], d1+d2)
        push!(Ms, Mpar)
        x = @variable(modelo, binary = true)
        @constraint(modelo, s[j1,k1] + d1 <= s[j2,k2] + Mpar*(1-x))
        @constraint(modelo, s[j2,k2] + d2 <= s[j1,k1] + Mpar*x)
    end

    @objective(modelo, Min, Cmax)
    optimize!(modelo)

    ok  = has_values(modelo)
    mk  = ok ? round(Int, objective_value(modelo)) : missing
    opt = otimos[nome]
    dist = ok ? round(100*(mk-opt)/opt, digits=1) : missing
    g   = relative_gap(modelo)
    gap = (ok && isfinite(g)) ? round(100*g, digits=1) : missing
    return (nome=nome, modo=modo, n=n, m=m, makespan=mk, otimo=opt, dist=dist,
            gap=gap, M=maximum(Ms), tempo=round(solve_time(modelo),digits=2),
            status=termination_status(modelo))
end

resolver (generic function with 1 method)

In [29]:
res_frouxo = []
for (i,nome) in enumerate(["ft20","ft10","abz5","orb01"])
    println("(",i,"/4) ",nome," [frouxo]..."); flush(stdout)
    r = resolver(nome; tempo_limite = 240, modo = "frouxo")
    push!(res_frouxo, r)
    println("   -> mk=",r.makespan," | +",r.dist,"% | gap=",r.gap,"% | ",r.tempo,"s | ",r.status)
    flush(stdout)
end

(1/4) ft20 [frouxo]...
   -> mk=1265 | +8.6% | gap=62.1% | 240.01s | TIME_LIMIT
(2/4) ft10 [frouxo]...
   -> mk=970 | +4.3% | gap=23.5% | 240.01s | TIME_LIMIT
(3/4) abz5 [frouxo]...
   -> mk=1239 | +0.4% | gap=16.6% | 240.01s | TIME_LIMIT
(4/4) orb01 [frouxo]...
   -> mk=1113 | +5.1% | gap=30.0% | 240.01s | TIME_LIMIT
